# CNN_VIT_BILSTM_CROSS_ATTENTION_BASED_TRAFFIC_MANAGEMENT_SYSTEM
## S14 — joint BMD-45 + IDD detector training

**Why this run exists.** S11 fine-tuned YOLOv8s on IDD and scored **mAP50 0.6201**
on IDD's own dashcam test split. Evaluated unchanged on 498 labelled images of
elevated Bengaluru CCTV, the same weights score **0.3223** — and find only
**39% of the vehicles present**, with the shortfall *growing* with density
(0.480 of vehicles in the sparsest third, 0.368 in the densest).

That last number is why this is the critical path rather than an improvement. A
constant shortfall could be corrected by moving the §14.1 count thresholds. One
that worsens with density compresses the MED/HIGH distinction the congestion
label depends on, so ADR-002's auto-labelling would come out **biased toward
"less congested"** — the worst available direction for a congestion predictor.

**Two sources, one training set, two separate test sets.**

| | |
|---|---|
| **BMD-45** | 3,679 Safe City CCTV cameras, Bengaluru. CC BY 4.0. The deployment viewpoint |
| **IDD** | car-mounted rig. The **only** source of `pedestrian` and `cattle` |

They are trained **jointly, never sequentially**. Fine-tuning on IDD last would
end training on a 100% dashcam corpus and un-teach the geometry this run exists
to learn — the last dataset a model sees is the one it believes.

**The test splits are NOT merged**, and that is deliberate. A union test set
reports one number dominated by whichever source contributed more images, so a
model could improve on dashcam, get *worse* on CCTV, and post a higher headline.
Every number below names which config produced it.

**Seed 42 throughout** (NFR-07). Results land in CSVs that get committed.

In [ ]:
import os, sys, random
SEED = 42
random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

import numpy as np, torch
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("NO GPU. Settings -> Accelerator -> GPU T4 x2.")
print("gpu  ", torch.cuda.get_device_name(0))

# Kaggle's P100 is sm_60 and current PyTorch starts at sm_70 — the card is
# UNUSABLE, not merely slow, and the only hint is a warning that scrolls past.
# Fail in fifteen seconds rather than after an hour of quota.
major, minor = torch.cuda.get_device_capability(0)
supported = torch.cuda.get_arch_list()
print("arch  ", f"sm_{major}{minor}", "| build supports", supported)
if f"sm_{major}{minor}" not in supported:
    raise SystemExit(
        f"INCOMPATIBLE GPU sm_{major}{minor}; build supports {supported}. "
        f"Settings -> Accelerator -> GPU T4 x2 (sm_75)."
    )

## The code comes from the repository, not from this notebook

Everything below runs **committed, tested scripts**. A notebook that reimplements
the conversion is a second implementation that will drift from the first, and the
S11 metrics defect showed what a small divergence costs. This is also NFR-08:
the run is reproducible from a clean machine because the machine fetches the same
code anyone else would.

In [ ]:
!git clone --depth 1 https://github.com/Divyansh-9/CNN_VIT_BILSTM_CROSS_ATTENTION_BASED_TRAFFIC_MANAGEMENT_SYSTEM.git /kaggle/working/repo 2>&1 | tail -2
%cd /kaggle/working/repo
!pip install -q ultralytics huggingface_hub
!python -m pytest tests/test_idd_mapping.py -q 2>&1 | tail -3

## BMD-45

Pulled from the Hub inside Kaggle, where bandwidth and disk are free. Serial
fetching runs at ~6 images/minute because the cost is per-request latency, not
throughput; `--workers` makes it roughly twelve times that.

`--count` is matched to the IDD subsample so neither source silently dominates
the union.

In [ ]:
COUNT = 8000     # matched to the IDD subsample
import subprocess

# --workers 16 through huggingface_hub, NOT raw HTTP at 32.
#
# Version 2 fetched with 32 raw urllib workers and lost 4,996 of 8,000
# images in 153 seconds — the Hub throttles anonymous raw requests. It then
# trained for 2.6 hours on the remainder before the metrics step refused to
# report a support of zero. The client negotiates the CDN and handles
# rate-limit responses itself.
#
# The failure-rate gate now aborts above 2%, so a repeat stops in the third
# minute. Completed images are kept and skipped, so a re-run resumes.
subprocess.run([
    "python", "scripts/prepare_bmd45.py",
    "--count", str(COUNT), "--workers", "16",
    "--max-failure-rate", "0.02",
    "--out", "/kaggle/working/bmd45_yolo",
], check=True)

## IDD

Attached as a Kaggle dataset. It mounts at its own slug path and nests, so the
directory is **discovered** rather than asserted — a hardcoded `/kaggle/input/...`
killed two earlier runs and the error said only that the path was missing, not
what was actually there.

In [ ]:
from pathlib import Path
INPUT = Path("/kaggle/input")
candidates = sorted(INPUT.iterdir()) if INPUT.exists() else []
print("mounted:", [c.name for c in candidates] or "NOTHING")
if not candidates:
    raise SystemExit(
        "Nothing under /kaggle/input. Attach 'indiatrafficnet-bootstrap-idd-yolo' "
        "via the Input panel — a UI Save & Run All uses the draft's attachments, "
        "not kernel-metadata's dataset_sources."
    )

def find_yolo_root(base):
    for path in [base, *base.rglob("*")]:
        if path.is_dir() and (path / "images" / "train").is_dir():
            return path
    return None

IDD = next((r for c in candidates if (r := find_yolo_root(c))), None)
if IDD is None:
    raise SystemExit(f"no images/train under any of {[c.name for c in candidates]}")
print("IDD  ", IDD)

## Compose — one training set, two evaluation sets

Nothing is copied. Ultralytics accepts a list of image directories, so the union
is expressed in a config file rather than by duplicating gigabytes.

Read the census below before reading any mAP. `pedestrian` and `cattle` exist in
**IDD only**, so the joint model learns them at the wrong viewpoint and their
elevated performance is unvalidated — there are no elevated labels for them to
be validated against.

In [ ]:
import subprocess
subprocess.run([
    "python", "scripts/build_joint_dataset.py",
    "--bmd45", "/kaggle/working/bmd45_yolo",
    "--idd", str(IDD),
    "--out", "/kaggle/working/joint",
], check=True)

## Train

`yolov8s` per PRD §12.3 — not `n` (too weak for small three-wheelers at distance,
which is exactly the failure mode being fixed) and not `m` (misses FR-D06's
≥10 fps edge budget).

Same recipe as S11 so the comparison is about **data, not hyperparameters**. If
this run changed the model size or the schedule as well, an improvement could not
be attributed to the elevated data.

In [ ]:
from ultralytics import YOLO

# A31 — GEOMETRIC AUGMENTATION, and it is the point of this rerun.
#
# S14 used Ultralytics defaults: perspective 0.0, degrees 0.0, shear 0.0.
# Every augmentation that teaches viewpoint invariance was off, and the
# detector lost 56% of its detections when the camera pitch was raised
# toward vertical. Barrel distortion over a comparable range cost only
# 17%, so ANGLE is the variable, not the lens.
#
# The deployment camera is an elevated fixed camera whose exact angle we
# do not control, so this is a requirement rather than a nicety.
model = YOLO("yolov8s.pt")
results = model.train(
    data="/kaggle/working/joint/joint.yaml",
    epochs=60, imgsz=640, batch=16, seed=SEED, patience=20,
    perspective=0.0006,   # Ultralytics range is 0 to 0.001
    degrees=8.0,
    shear=4.0,
    project="/kaggle/working/runs", name="s14_joint", exist_ok=True, plots=True,
)

## Evaluate — separately, and against S11

`scripts/verify_detector_metrics.py` keys every row by `ap_class_index`. Ultralytics'
`class_result(i)` indexes into classes that **have instances**, not into the names
list, so a class with zero boxes shifts every row after it. S11's first published
table reported mAP50 0.7288 for a class with 0 boxes and `nan` for one with 183.

BMD-45 carries no `pedestrian`, `cattle` or `e_rickshaw`, so three rows will read
NOT EVALUATED on the elevated config. That is correct, not a failure.

In [ ]:
WEIGHTS = "/kaggle/working/runs/s14_joint/weights/best.pt"

# subprocess, not `!` — the shell magic substitutes {expressions} from the
# user namespace and the rules for what it accepts are subtle. An argument
# list has no quoting or substitution semantics to get wrong.
import subprocess

BAR = "=================================================================="
for tag, what in (("bmd45", "elevated CCTV"), ("idd", "dashcam")):
    print()
    print(BAR)
    print(f"  {tag.upper()}  ({what})")
    print(BAR)
    subprocess.run([
        "python", "scripts/verify_detector_metrics.py",
        "--weights", WEIGHTS,
        "--data", f"/kaggle/working/joint/eval_{tag}.yaml",
        "--split", "test", "--device", "0",
        "--out", f"/kaggle/working/s14_metrics_{tag}.csv",
    ], check=True)

## Counting accuracy — FR-P02

Detection mAP does not answer this. mAP is a per-class ranking measure over IoU
thresholds; a count is one integer per frame, and a detector can hold respectable
mAP while losing the same fraction of vehicles in every frame.

**The number to watch is the density split.** S11 detected 0.480 of vehicles in
the sparsest third and 0.368 in the densest. If that gap closes, the §14.1
thresholds become usable on elevated footage. If it does not, auto-labelling stays
prohibited there regardless of what mAP says.

In [ ]:
import subprocess
ROOTS = {"bmd45": "/kaggle/working/bmd45_yolo", "idd": str(IDD)}
for tag, root in ROOTS.items():
    print()
    print("=====", tag, "=====")
    subprocess.run([
        "python", "scripts/verify_counting.py",
        "--weights", WEIGHTS, "--data", root,
        "--split", "test", "--device", "0",
        "--out", f"/kaggle/working/s14_counting_{tag}.csv",
    ], check=True)

## Export

Download the CSVs and `best.pt`. Weights go to Hugging Face, not git (ADR-013
rev 2) — LFS bandwidth is spent by every clone and CI checkout, and exhausting it
reads to a teammate as a broken clone rather than as a quota.

In [ ]:
best = Path("/kaggle/working/runs/s14_joint/weights/best.pt")
print("weights", best, round(best.stat().st_size/1e6, 1), "MB" if best.exists() else "MISSING")
print()
print("S11 baseline, for the comparison this run exists to make:")
print("   dashcam  mAP50 0.6201   elevated mAP50 0.3223")
print("   elevated counting: 0.391 of vehicles; 0.480 sparse -> 0.368 dense")
print()
print("Commit s14_metrics_*.csv and s14_counting_*.csv to experiments/results/.")
print("Report BOTH configs and name which produced each number. If dashcam mAP")
print("fell, say so — that trade is a result, not something to leave out.")